# 04 - Análise de Dados

Este notebook utiliza as tabelas da camada Gold para responder às perguntas de negócio definidas no objetivo do MVP.

As análises consideram indicadores de atrasos, cancelamentos, companhias aéreas, aeroportos, rotas e comportamento temporal das operações.

In [0]:
fato_voo = spark.table("workspace.mvp_sprint_3_anac.gold_fato_voo")
dim_companhia = spark.table("workspace.mvp_sprint_3_anac.gold_dim_companhia")
dim_aeroporto = spark.table("workspace.mvp_sprint_3_anac.gold_dim_aeroporto")
dim_rota = spark.table("workspace.mvp_sprint_3_anac.gold_dim_rota")
dim_tempo = spark.table("workspace.mvp_sprint_3_anac.gold_dim_tempo")

print("Fato voo:", fato_voo.count())

Fato voo: 591431


In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    round as spark_round,
    when
)

analise_aeroportos = (
    fato_voo
    .groupBy("codigo_aeroporto_origem")
    .agg(
        count("*").alias("total_voos"),

        spark_sum(
            when(col("atraso_partida_min").isNotNull(), 1).otherwise(0)
        ).alias("voos_com_atraso_calculavel"),

        spark_sum(
            when(col("flag_atraso_30") == 1, 1).otherwise(0)
        ).alias("voos_atraso_30"),

        spark_sum("flag_cancelado").alias("voos_cancelados")
    )
    .withColumn(
        "taxa_atraso_30_pct",
        spark_round(
            100
            * col("voos_atraso_30")
            / col("voos_com_atraso_calculavel"),
            2
        )
    )
    .withColumn(
        "taxa_cancelamento_pct",
        spark_round(
            100
            * col("voos_cancelados")
            / col("total_voos"),
            2
        )
    )
)

In [0]:
top_aeroportos_atraso = (
    analise_aeroportos
    .filter(col("voos_com_atraso_calculavel") >= 500)
    .orderBy(col("taxa_atraso_30_pct").desc())
)

display(top_aeroportos_atraso)

codigo_aeroporto_origem,total_voos,voos_com_atraso_calculavel,voos_atraso_30,voos_cancelados,taxa_atraso_30_pct,taxa_cancelamento_pct
KMIA,3853,2806,805,359,28.69,9.32
SLVR,649,541,149,77,27.54,11.86
LEMD,2946,1725,444,1146,25.74,38.9
LPPT,3484,3403,718,44,21.1,1.26
LFPG,1081,1060,209,6,19.72,0.56
EHAM,746,578,113,141,19.55,18.9
LIRF,978,779,145,158,18.61,16.16
KJFK,818,780,138,27,17.69,3.3
SAEZ,5365,3577,565,1638,15.8,30.53
SABE,6494,6107,838,134,13.72,2.06


In [0]:
top_aeroportos_cancelamento = (
    analise_aeroportos
    .filter(col("total_voos") >= 500)
    .orderBy(col("taxa_cancelamento_pct").desc())
)

display(top_aeroportos_cancelamento)

codigo_aeroporto_origem,total_voos,voos_com_atraso_calculavel,voos_atraso_30,voos_cancelados,taxa_atraso_30_pct,taxa_cancelamento_pct
MDPC,1074,340,121,717,35.59,66.76
DNMM,700,248,74,394,29.84,56.29
OTHH,765,395,90,330,22.78,43.14
LEMD,2946,1725,444,1146,25.74,38.9
SAEZ,5365,3577,565,1638,15.8,30.53
SEQM,573,233,75,119,32.19,20.77
EHAM,746,578,113,141,19.55,18.9
GVAC,567,133,84,107,63.16,18.87
LIRF,978,779,145,158,18.61,16.16
SACO,976,809,91,155,11.25,15.88


In [0]:
top_atraso_com_nome = (
    top_aeroportos_atraso
    .join(
        dim_aeroporto.select(
            col("codigo_oaci").alias("codigo_aeroporto_origem"),
            "descricao_aeroporto"
        ),
        on="codigo_aeroporto_origem",
        how="left"
    )
    .select(
        "codigo_aeroporto_origem",
        "descricao_aeroporto",
        "total_voos",
        "voos_com_atraso_calculavel",
        "voos_atraso_30",
        "taxa_atraso_30_pct",
        "voos_cancelados",
        "taxa_cancelamento_pct"
    )
    .orderBy(col("taxa_atraso_30_pct").desc())
    .limit(10)
)

display(top_atraso_com_nome)

codigo_aeroporto_origem,descricao_aeroporto,total_voos,voos_com_atraso_calculavel,voos_atraso_30,taxa_atraso_30_pct,voos_cancelados,taxa_cancelamento_pct
KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA - ESTADOS UNIDOS DA AMÉRICA",3853,2806,805,28.69,359,9.32
SLVR,VIRU VIRU INTERNATIONAL AIRPORT - SANTA CRUZ - BOLÍVIA,649,541,149,27.54,77,11.86
LEMD,ADOLFO SUÁREZ MADRID-BARAJAS AIRPORT - MADRID - ESPANHA,2946,1725,444,25.74,1146,38.9
LPPT,LISBOA - LISBOA - PORTUGAL,3484,3403,718,21.1,44,1.26
LFPG,PARIS-CHARLES DE GAULLE AIRPORT (ROISSY AIRPORT) - PARIS - FRANÇA,1081,1060,209,19.72,6,0.56
EHAM,"AMSTERDAM AIRPORT SCHIPHOL - HAARLEMMERMEER, NEAR AMSTERDAM - HOLANDA",746,578,113,19.55,141,18.9
LIRF,LEONARDO DA VINCI INTERNATIONAL AIRPORT (FIUMICINO INTERNATIONAL AIRPORT) - ROME - ITÁLIA,978,779,145,18.61,158,16.16
KJFK,"JOHN F. KENNEDY INTERNATIONAL AIRPORT - NEW YORK, NEW YORK - ESTADOS UNIDOS DA AMÉRICA",818,780,138,17.69,27,3.3
SAEZ,"MINISTRO PISTARINI INTERNATIONAL AIRPORT (EZEIZA INTERNATIONAL AIRPORT) - EZEIZA, BUENOS AIRES PROVINCE - ARGENTINA",5365,3577,565,15.8,1638,30.53
SABE,JORGE NEWBERY AIRPORT - BUENOS AIRES - ARGENTINA,6494,6107,838,13.72,134,2.06


In [0]:
top_cancelamento_com_nome = (
    top_aeroportos_cancelamento
    .join(
        dim_aeroporto.select(
            col("codigo_oaci").alias("codigo_aeroporto_origem"),
            "descricao_aeroporto"
        ),
        on="codigo_aeroporto_origem",
        how="left"
    )
    .select(
        "codigo_aeroporto_origem",
        "descricao_aeroporto",
        "total_voos",
        "voos_com_atraso_calculavel",
        "voos_atraso_30",
        "taxa_atraso_30_pct",
        "voos_cancelados",
        "taxa_cancelamento_pct"
    )
    .orderBy(col("taxa_cancelamento_pct").desc())
    .limit(10)
)

display(top_cancelamento_com_nome)

codigo_aeroporto_origem,descricao_aeroporto,total_voos,voos_com_atraso_calculavel,voos_atraso_30,taxa_atraso_30_pct,voos_cancelados,taxa_cancelamento_pct
MDPC,PUNTA CANA - PUNTA CANA - REPÚBLICA DOMINICANA,1074,340,121,35.59,717,66.76
DNMM,"MURTALA MOHAMMED INTERNATIONAL AIRPORT - IKEJA, LAGOS STATE - NIGÉRIA",700,248,74,29.84,394,56.29
OTHH,HAMAD INTERNATIONAL AIRPORT - DOHA - QATAR,765,395,90,22.78,330,43.14
LEMD,ADOLFO SUÁREZ MADRID-BARAJAS AIRPORT - MADRID - ESPANHA,2946,1725,444,25.74,1146,38.9
SAEZ,"MINISTRO PISTARINI INTERNATIONAL AIRPORT (EZEIZA INTERNATIONAL AIRPORT) - EZEIZA, BUENOS AIRES PROVINCE - ARGENTINA",5365,3577,565,15.8,1638,30.53
SEQM,MARISCAL SUCRE INTERNATIONAL AIRPORT - QUITO - EQUADOR,573,233,75,32.19,119,20.77
EHAM,"AMSTERDAM AIRPORT SCHIPHOL - HAARLEMMERMEER, NEAR AMSTERDAM - HOLANDA",746,578,113,19.55,141,18.9
GVAC,AEROPORTO INTERNACIONAL AMÍLCAR CABRAL - ILHA DO SAL - CABO VERDE,567,133,84,63.16,107,18.87
LIRF,LEONARDO DA VINCI INTERNATIONAL AIRPORT (FIUMICINO INTERNATIONAL AIRPORT) - ROME - ITÁLIA,978,779,145,18.61,158,16.16
SACO,"INGENIERO AMBROSIO L.V. TARAVELLA INTERNATIONAL AIRPORT - CÓRDOBA, CÓRDOBA PROVINCE - ARGENTINA",976,809,91,11.25,155,15.88


## Pergunta 1 - Quais aeroportos apresentam as maiores taxas de atrasos superiores a 30 minutos e de cancelamentos?

Para reduzir distorções causadas por aeroportos com pequeno número de operações válidas para o cálculo de atraso, foram considerados, no ranking de atrasos, apenas aeroportos de origem com pelo menos 500 voos para os quais foi possível calcular a diferença entre os horários previsto e realizado. Para o ranking de cancelamentos, foram considerados aeroportos com pelo menos 500 registros de voos no período.

Entre os aeroportos analisados, Miami (KMIA) apresentou a maior taxa de partidas com atraso superior a 30 minutos, com 28,69% dos voos com atraso calculável. Em seguida apareceram Viru Viru (SLVR), com 27,54%, Madrid-Barajas (LEMD), com 25,74%, Lisboa (LPPT), com 21,10%, e Paris-Charles de Gaulle (LFPG), com 19,72%.

Em relação aos cancelamentos, Punta Cana (MDPC) apresentou o maior percentual, com 66,76% dos registros cancelados. Também foram observadas taxas elevadas em Murtala Muhammed/Lagos (DNMM), com 56,29%, Hamad/Doha (OTHH), com 43,14%, Madrid-Barajas (LEMD), com 38,90%, e Ezeiza/Buenos Aires (SAEZ), com 30,53%.

Os resultados mostram que atraso e cancelamento representam dimensões distintas do desempenho operacional. Um aeroporto pode apresentar elevada incidência de atrasos entre os voos efetivamente operados e, ao mesmo tempo, uma taxa de cancelamento diferente. O conjunto VRA também inclui operações envolvendo aeroportos internacionais, razão pela qual vários deles aparecem entre os maiores percentuais encontrados.

In [0]:
analise_companhias = (
    fato_voo
    .groupBy("sigla_icao_empresa_aerea")
    .agg(
        count("*").alias("total_voos"),

        spark_sum(
            when(col("atraso_partida_min").isNotNull(), 1).otherwise(0)
        ).alias("voos_com_atraso_calculavel"),

        spark_sum(
            when(col("flag_atraso_30") == 1, 1).otherwise(0)
        ).alias("voos_atraso_30"),

        spark_sum("flag_cancelado").alias("voos_cancelados")
    )
    .withColumn(
        "taxa_atraso_30_pct",
        spark_round(
            100 *
            col("voos_atraso_30") /
            col("voos_com_atraso_calculavel"),
            2
        )
    )
    .withColumn(
        "taxa_cancelamento_pct",
        spark_round(
            100 *
            col("voos_cancelados") /
            col("total_voos"),
            2
        )
    )
)

In [0]:
companhias_atraso = (
    analise_companhias
    .filter(col("voos_com_atraso_calculavel") >= 500)
    .join(
        dim_companhia,
        on="sigla_icao_empresa_aerea",
        how="left"
    )
)

display(
    companhias_atraso
    .orderBy(col("taxa_atraso_30_pct").asc())
    .limit(10)
)

sigla_icao_empresa_aerea,total_voos,voos_com_atraso_calculavel,voos_atraso_30,voos_cancelados,taxa_atraso_30_pct,taxa_cancelamento_pct,empresa_aerea
ABJ,803,694,11,90,1.59,11.21,ATA - AEROTÁXI ABAETÉ LTDA.
CMP,5898,5623,243,258,4.32,4.37,COMPAÑIA PANAMEÑA DE AVIACION S.A. (COPA AIRLINES)
GLO,151379,148091,9229,1746,6.23,1.15,GOL LINHAS AÉREAS S.A. (EX- VRG LINHAS AÉREAS S.A.)
SWR,556,529,33,16,6.24,2.88,SWISS INTERNATIONAL AIR LINES LTD.
BAW,1277,1253,80,19,6.38,1.49,BRITISH AIRWAYS PLC
AZU,163332,156926,12895,2786,8.22,1.71,AZUL LINHAS AÉREAS BRASILEIRAS S/A
TAM,175619,170828,14571,2360,8.53,1.34,TAM LINHAS AÉREAS S.A.
LPE,2878,2813,247,21,8.78,0.73,LATAM AIRLINES PERU (EX-LAN PERU S.A.)
LAN,8342,7761,720,58,9.28,0.7,LATAM AIRLINES GROUP (EX - LAN AIRLINES S/A)
JAT,2702,2378,221,60,9.29,2.22,JETSMART AIRLINES SPA - CHILE


In [0]:
companhias_cancelamento = (
    analise_companhias
    .filter(col("total_voos") >= 500)
    .join(
        dim_companhia,
        on="sigla_icao_empresa_aerea",
        how="left"
    )
)

display(
    companhias_cancelamento
    .orderBy(col("taxa_cancelamento_pct").asc())
    .limit(10)
)

sigla_icao_empresa_aerea,total_voos,voos_com_atraso_calculavel,voos_atraso_30,voos_cancelados,taxa_atraso_30_pct,taxa_cancelamento_pct,empresa_aerea
LAP,751,739,113,4,15.29,0.53,TRANSPORTE AÉREOS DEL MERCOSUR S.A. (TAM MERCOSUR)
AFR,1849,1813,300,12,16.55,0.65,SOCIÉTÉ AIR FRANCE
AVA,3215,3170,382,22,12.05,0.68,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA
LAN,8342,7761,720,58,9.28,0.7,LATAM AIRLINES GROUP (EX - LAN AIRLINES S/A)
LPE,2878,2813,247,21,8.78,0.73,LATAM AIRLINES PERU (EX-LAN PERU S.A.)
GLO,151379,148091,9229,1746,6.23,1.15,GOL LINHAS AÉREAS S.A. (EX- VRG LINHAS AÉREAS S.A.)
TAP,6002,5846,974,76,16.66,1.27,TAP - TRANSPORTES AÉREOS PORTUGUESES S/A
THY,1888,1837,274,25,14.92,1.32,TURKISH AIRLINES INC
TAM,175619,170828,14571,2360,8.53,1.34,TAM LINHAS AÉREAS S.A.
ARE,872,836,104,12,12.44,1.38,AIRES - AEROVÍAS DE INTEGRACÍON REGIONAL S.A.


## Pergunta 2 - Quais companhias aéreas apresentam os menores percentuais de atrasos superiores a 30 minutos e de cancelamentos?

Para reduzir distorções causadas por companhias com baixo volume de operações, foram utilizados critérios mínimos de volume compatíveis com cada indicador. No ranking de atrasos, foram consideradas apenas companhias com pelo menos 500 voos para os quais foi possível calcular o atraso de partida. No ranking de cancelamentos, foram consideradas companhias com pelo menos 500 registros de voos no período.

Em relação aos atrasos superiores a 30 minutos, as menores taxas foram observadas para as companhias de código ICAO ABJ (1,59%), CMP (4,32%), GLO (6,23%), SWR (6,24%) e BAW (6,38%). Entre as companhias com maior volume de operações no conjunto analisado, GLO apresentou 6,23%, AZU 8,22% e TAM 8,53%.

Para cancelamentos, os menores percentuais observados foram LAP (0,53%), AFR (0,65%), AVA (0,68%), LAN (0,70%) e LPE (0,73%). GLO apresentou taxa de cancelamento de 1,15%, enquanto TAM apresentou 1,34%.

Os resultados mostram que desempenho em atraso e cancelamento não deve ser interpretado como uma única métrica. Uma companhia pode apresentar baixa taxa de cancelamento e, ao mesmo tempo, possuir uma proporção relativamente maior de atrasos entre os voos efetivamente operados. Por esse motivo, os dois indicadores foram avaliados separadamente.

In [0]:
from pyspark.sql.functions import percentile_approx, avg

fato_com_tempo = (
    fato_voo
    .join(
        dim_tempo.select(
            "data",
            "dia_semana_num",
            "dia_semana"
        ),
        on="data",
        how="left"
    )
)

In [0]:
analise_dia_semana = (
    fato_com_tempo
    .filter(col("atraso_partida_min").isNotNull())
    .groupBy(
        "dia_semana_num",
        "dia_semana"
    )
    .agg(
        count("*").alias("voos_analisados"),

        spark_sum(
            when(col("flag_atraso_30") == 1, 1).otherwise(0)
        ).alias("voos_atraso_30"),

        spark_round(
            avg(
                when(
                    col("flag_atraso_30") == 1,
                    col("atraso_partida_min")
                )
            ),
            2
        ).alias("media_atraso_30_min"),

        percentile_approx(
            when(
                col("flag_atraso_30") == 1,
                col("atraso_partida_min")
            ),
            0.5
        ).alias("mediana_atraso_30_min")
    )
    .withColumn(
        "taxa_atraso_30_pct",
        spark_round(
            100 *
            col("voos_atraso_30") /
            col("voos_analisados"),
            2
        )
    )
    .orderBy("dia_semana_num")
)

display(analise_dia_semana)

dia_semana_num,dia_semana,voos_analisados,voos_atraso_30,media_atraso_30_min,mediana_atraso_30_min,taxa_atraso_30_pct
1,Sunday,75665,6355,109.78,55.0,8.4
2,Monday,80499,6619,91.49,53.0,8.22
3,Tuesday,79927,6452,90.92,53.0,8.07
4,Wednesday,80696,7452,90.27,53.0,9.23
5,Thursday,83167,8526,93.17,55.0,10.25
6,Friday,82988,8283,81.39,52.0,9.98
7,Saturday,73375,5946,101.99,57.0,8.1


In [0]:
analise_periodo_dia = (
    fato_voo
    .filter(col("atraso_partida_min").isNotNull())
    .groupBy("periodo_dia")
    .agg(
        count("*").alias("voos_analisados"),

        spark_sum(
            when(col("flag_atraso_30") == 1, 1).otherwise(0)
        ).alias("voos_atraso_30"),

        spark_round(
            avg(
                when(
                    col("flag_atraso_30") == 1,
                    col("atraso_partida_min")
                )
            ),
            2
        ).alias("media_atraso_30_min"),

        percentile_approx(
            when(
                col("flag_atraso_30") == 1,
                col("atraso_partida_min")
            ),
            0.5
        ).alias("mediana_atraso_30_min")
    )
    .withColumn(
        "taxa_atraso_30_pct",
        spark_round(
            100 *
            col("voos_atraso_30") /
            col("voos_analisados"),
            2
        )
    )
    .orderBy(col("taxa_atraso_30_pct").desc())
)

display(analise_periodo_dia)

periodo_dia,voos_analisados,voos_atraso_30,media_atraso_30_min,mediana_atraso_30_min,taxa_atraso_30_pct
Noite,139159,16492,87.76,52.0,11.85
Tarde,181137,16689,84.86,53.0,9.21
Madrugada,44337,3625,132.46,60.0,8.18
Manhã,191684,12827,100.86,56.0,6.69


## Pergunta 3 - Como a frequência e a duração dos atrasos variam conforme o dia da semana e o período do dia?

A frequência dos atrasos foi calculada considerando os voos para os quais foi possível obter a diferença entre os horários previsto e realizado. Para avaliar a duração dos atrasos, a média e a mediana foram calculadas somente entre os voos com atraso superior a 30 minutos.

Por dia da semana, a maior incidência de atrasos superiores a 30 minutos ocorreu às quintas-feiras, com 10,25% dos voos analisados, seguida pelas sextas-feiras, com 9,98%, e pelas quartas-feiras, com 9,23%. As menores taxas foram observadas nas terças-feiras (8,07%), sábados (8,10%) e segundas-feiras (8,22%).

A duração dos atrasos apresenta um comportamento diferente da frequência. Entre os voos com atraso superior a 30 minutos, o domingo apresentou a maior média, com 109,78 minutos, seguido pelo sábado, com 101,99 minutos. A sexta-feira apresentou a menor média, com 81,39 minutos. As medianas variaram de 52 a 57 minutos, indicando que metade dos atrasos superiores a 30 minutos ficou aproximadamente dentro dessa faixa.

Por período do dia, a noite apresentou a maior frequência de atrasos superiores a 30 minutos, com 11,85%, seguida pela tarde (9,21%), madrugada (8,18%) e manhã (6,69%). Entretanto, os atrasos ocorridos durante a madrugada foram, em média, os mais longos, alcançando 132,46 minutos, com mediana de 60 minutos. A noite apresentou média de 87,76 minutos e mediana de 52 minutos.

Os resultados indicam que frequência e duração não seguem necessariamente o mesmo padrão. O período noturno concentra a maior proporção de voos atrasados, enquanto a madrugada, apesar de apresentar menor frequência, concentra atrasos mais longos entre os casos que ultrapassam 30 minutos.

In [0]:
analise_rotas = (
    fato_voo
    .groupBy(
        "rota_id",
        "codigo_aeroporto_origem",
        "codigo_aeroporto_destino"
    )
    .agg(
        count("*").alias("total_voos"),

        spark_sum(
            when(col("atraso_partida_min").isNotNull(), 1).otherwise(0)
        ).alias("voos_com_atraso_calculavel"),

        spark_sum(
            when(col("flag_atraso_30") == 1, 1).otherwise(0)
        ).alias("voos_atraso_30"),

        spark_sum("flag_cancelado").alias("voos_cancelados")
    )
    .withColumn(
        "taxa_atraso_30_pct",
        when(
            col("voos_com_atraso_calculavel") > 0,
            spark_round(
                100 *
                col("voos_atraso_30") /
                col("voos_com_atraso_calculavel"),
                2
            )
        ).otherwise(None)
    )
    .withColumn(
        "taxa_cancelamento_pct",
        spark_round(
            100 *
            col("voos_cancelados") /
            col("total_voos"),
            2
        )
    )
)

In [0]:
top_rotas_atraso = (
    analise_rotas
    .filter(col("voos_com_atraso_calculavel") >= 200)
    .orderBy(col("taxa_atraso_30_pct").desc())
    .limit(10)
)

display(top_rotas_atraso)

rota_id,codigo_aeroporto_origem,codigo_aeroporto_destino,total_voos,voos_com_atraso_calculavel,voos_atraso_30,voos_cancelados,taxa_atraso_30_pct,taxa_cancelamento_pct
SBGR->HAAB,SBGR,HAAB,215,211,131,1,62.09,0.47
CYYZ->SBGR,CYYZ,SBGR,245,241,96,2,39.83,0.82
MDPC->SBGR,MDPC,SBGR,370,340,121,17,35.59,4.59
KORD->SBGR,KORD,SBGR,214,212,72,2,33.96,0.93
KMIA->SBGL,KMIA,SBGL,365,339,115,4,33.92,1.1
LFPG->SBGL,LFPG,SBGL,258,250,80,1,32.0,0.39
SBGR->SAEZ,SBGR,SAEZ,924,858,271,20,31.59,2.16
KMIA->SBKP,KMIA,SBKP,668,504,156,41,30.95,6.14
KEWR->SBGR,KEWR,SBGR,214,208,64,3,30.77,1.4
LPPT->SBSV,LPPT,SBSV,203,201,61,1,30.35,0.49


In [0]:
top_rotas_cancelamento = (
    analise_rotas
    .filter(col("total_voos") >= 200)
    .orderBy(col("taxa_cancelamento_pct").desc())
    .limit(10)
)

display(top_rotas_cancelamento)

rota_id,codigo_aeroporto_origem,codigo_aeroporto_destino,total_voos,voos_com_atraso_calculavel,voos_atraso_30,voos_cancelados,taxa_atraso_30_pct,taxa_cancelamento_pct
SAEZ->LEMD,SAEZ,LEMD,640,0,0,640,null,100.0
SPJC->LEMD,SPJC,LEMD,252,0,0,252,null,100.0
LEMD->SPJC,LEMD,SPJC,247,0,0,247,null,100.0
SAEZ->MDPC,SAEZ,MDPC,514,0,0,514,null,100.0
LEMD->SAEZ,LEMD,SAEZ,649,0,0,649,null,100.0
MDPC->SAEZ,MDPC,SAEZ,492,0,0,492,null,100.0
HAAB->SBGR,HAAB,SBGR,210,69,16,131,23.19,62.38
DNMM->SBGR,DNMM,SBGR,684,241,67,386,27.8,56.43
SBGR->OTHH,SBGR,OTHH,624,421,57,194,13.54,31.09
SBGR->SEQM,SBGR,SEQM,265,140,30,80,21.43,30.19


## Pergunta 4 - Quais rotas apresentam os maiores percentuais de atrasos e cancelamentos?

Para reduzir distorções causadas por rotas com poucas observações válidas, o ranking de atrasos considerou apenas rotas com pelo menos 200 voos para os quais foi possível calcular o atraso de partida. Para o ranking de cancelamentos, foram consideradas rotas com pelo menos 200 registros de voos no período. As rotas foram tratadas de forma direcional, portanto origem → destino e destino → origem representam rotas distintas.

Entre as rotas analisadas, SBGR → HAAB apresentou a maior taxa de atrasos superiores a 30 minutos, com 62,09%. Em seguida apareceram CYYZ → SBGR, com 39,83%, MDPC → SBGR, com 35,59%, KORD → SBGR, com 33,96%, e KMIA → SBGL, com 33,92%.

Na análise de cancelamentos, foram identificadas rotas em que 100% dos registros do período estavam classificados como cancelados, entre elas SAEZ → LEMD, SPJC → LEMD, LEMD → SPJC, SAEZ → MDPC, LEMD → SAEZ e MDPC → SAEZ. Nessas situações não existem voos com atraso calculável, pois não há horário de partida realizado para os registros cancelados.

Os resultados reforçam a necessidade de analisar atrasos e cancelamentos separadamente. Uma rota totalmente cancelada pode apresentar ausência de atrasos calculáveis, mas isso não representa bom desempenho operacional. Da mesma forma, rotas com baixa taxa de cancelamento podem apresentar elevada incidência de atrasos entre os voos efetivamente realizados.

Os percentuais encontrados representam exclusivamente o recorte de janeiro a julho de 2026 presente no conjunto VRA analisado e não devem ser interpretados como características permanentes das rotas.

In [0]:
analise_mensal = (
    fato_voo
    .join(
        dim_tempo.select(
            "data",
            "ano",
            "mes"
        ),
        on="data",
        how="left"
    )
    .groupBy(
        "ano",
        "mes"
    )
    .agg(
        count("*").alias("total_voos"),

        spark_sum(
            when(col("atraso_partida_min").isNotNull(), 1).otherwise(0)
        ).alias("voos_com_atraso_calculavel"),

        spark_sum(
            when(col("flag_atraso_30") == 1, 1).otherwise(0)
        ).alias("voos_atraso_30"),

        spark_sum("flag_cancelado").alias("voos_cancelados"),

        spark_round(
            avg("atraso_partida_min"),
            2
        ).alias("media_atraso_min"),

        percentile_approx(
            "atraso_partida_min",
            0.5
        ).alias("mediana_atraso_min")
    )
    .withColumn(
        "taxa_atraso_30_pct",
        when(
            col("voos_com_atraso_calculavel") > 0,
            spark_round(
                100 *
                col("voos_atraso_30") /
                col("voos_com_atraso_calculavel"),
                2
            )
        )
    )
    .withColumn(
        "taxa_cancelamento_pct",
        spark_round(
            100 *
            col("voos_cancelados") /
            col("total_voos"),
            2
        )
    )
    .orderBy(
        "ano",
        "mes"
    )
)

display(analise_mensal)

ano,mes,total_voos,voos_com_atraso_calculavel,voos_atraso_30,voos_cancelados,media_atraso_min,mediana_atraso_min,taxa_atraso_30_pct,taxa_cancelamento_pct
2026,1,91650,86148,8436,2817,8.64,-1.0,9.79,3.07
2026,2,79455,74757,7042,2376,8.25,-1.0,9.42,2.99
2026,3,87198,82020,6596,2620,6.38,-3.0,8.04,3.0
2026,4,80381,75774,7368,2362,7.5,-2.0,9.72,2.94
2026,5,82120,77489,5784,2388,5.09,-2.0,7.46,2.91
2026,6,81119,75919,7295,2892,8.07,-1.0,9.61,3.57
2026,7,89508,84210,7112,2722,7.14,-1.0,8.45,3.04


## Pergunta 5 - Como os indicadores de pontualidade e cancelamento evoluíram entre janeiro e julho de 2026?

Os indicadores apresentaram variações ao longo dos sete meses analisados, sem uma tendência contínua de melhora ou piora.

A taxa de voos com atraso superior a 30 minutos foi de 9,79% em janeiro, 9,42% em fevereiro, 8,04% em março, 9,72% em abril, 7,46% em maio, 9,61% em junho e 8,45% em julho. O menor percentual ocorreu em maio, enquanto janeiro apresentou o maior valor entre os meses analisados.

O desvio médio do horário de partida em relação ao horário previsto também variou no período. A média foi de 8,64 minutos em janeiro, caiu para 8,25 em fevereiro e 6,38 em março, voltou a subir para 7,50 em abril e atingiu o menor valor em maio, com 5,09 minutos. Em junho houve nova elevação para 8,07 minutos, seguida de 7,14 minutos em julho.

As medianas permaneceram negativas durante todo o período, variando entre -1 e -3 minutos. Como essa métrica considera todos os voos com horários previsto e realizado disponíveis, valores negativos representam pequenas antecipações. A diferença entre medianas negativas e médias positivas indica uma distribuição assimétrica, na qual atrasos elevados em uma parcela menor das operações deslocam a média para valores positivos.

As taxas de cancelamento permaneceram relativamente estáveis, próximas de 3% na maior parte do período: 3,07% em janeiro, 2,99% em fevereiro, 3,00% em março, 2,94% em abril, 2,91% em maio, 3,57% em junho e 3,04% em julho. Junho apresentou o maior percentual de cancelamentos, enquanto maio apresentou o menor.

De forma geral, maio apresentou os menores indicadores de atraso e cancelamento no recorte analisado, enquanto junho registrou uma deterioração dos dois indicadores em relação ao mês anterior. Entretanto, os dados não demonstram uma tendência monotônica ao longo de janeiro a julho de 2026, mas sim oscilações mensais no desempenho operacional.

## Discussão Geral dos Resultados

As análises realizadas demonstraram que o desempenho operacional dos voos apresenta diferenças relevantes conforme aeroporto, companhia aérea, rota, período do dia e mês analisado.

Nos aeroportos, foram encontradas diferenças expressivas nas taxas de atraso e cancelamento, inclusive entre aeroportos com volumes significativos de operações. Como o VRA contempla voos internacionais relacionados à malha registrada pela ANAC, aeroportos localizados fora do Brasil também apareceram entre os maiores percentuais observados.

Na análise das companhias aéreas, verificou-se que atraso e cancelamento devem ser tratados como indicadores distintos. Empresas com baixas taxas de cancelamento não necessariamente apresentam as menores taxas de atraso, e o volume de operações também precisa ser considerado para evitar conclusões baseadas em amostras reduzidas.

A dimensão temporal mostrou que as diferenças entre períodos do dia são mais acentuadas do que as diferenças entre dias da semana. O período noturno apresentou a maior incidência de atrasos superiores a 30 minutos, enquanto a madrugada apresentou a maior duração média desses atrasos. O período da manhã apresentou a menor frequência de atrasos superiores a 30 minutos.

As rotas também apresentaram grande heterogeneidade. Algumas tiveram elevada incidência de atrasos entre os voos realizados, enquanto outras registraram 100% de cancelamento no recorte analisado. Esses casos reforçam a importância de manter atraso e cancelamento como métricas independentes.

Finalmente, a evolução mensal entre janeiro e julho de 2026 revelou oscilações, sem uma tendência contínua de melhora ou piora. Maio apresentou os menores indicadores de atraso e cancelamento, enquanto junho apresentou aumento em ambos.

De maneira geral, o pipeline desenvolvido permitiu transformar os arquivos brutos da ANAC em dados estruturados e confiáveis para análise. A organização em camadas Bronze, Silver e Gold possibilitou preservar os dados originais, tratar questões de qualidade, criar atributos derivados e disponibilizar uma modelagem dimensional adequada às perguntas formuladas no início do projeto.

Os resultados devem ser interpretados dentro do período e das fontes utilizadas. As análises identificam padrões e diferenças nos registros operacionais, mas não permitem estabelecer relações causais sobre os motivos de atrasos ou cancelamentos, uma vez que o conjunto de dados utilizado não contém informações suficientes sobre suas causas.